In [1]:
import sys
import os
import logging
import warnings
import random
import re
import gc
import time
import pandas as pd
from tqdm import tqdm

# ==========================================
# 0. 环境与依赖设置
# ==========================================
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("transformers").setLevel(logging.ERROR)

# 检查 Excel 导出依赖
try:
    import openpyxl
except ImportError:
    print("⚠️  'openpyxl' not found. Installing for Excel export...")
    os.system("pip install openpyxl")

# 检查 Unsloth
try:
    from unsloth import FastLanguageModel
    import torch
    HAS_UNSLOTH = True
    print("✅ Unsloth detected (Acceleration enabled).")
except ImportError:
    import torch
    HAS_UNSLOTH = False
    print("⚠️  Unsloth not found. Using standard Transformers.")

# 检查 OpenAI
try:
    from openai import OpenAI
except ImportError:
    print("⚠️  OpenAI lib not found. API models will be skipped.")

# ==========================================
# 1. 全局配置 (Configuration)
# ==========================================
CONFIG = {
    # ----------------------------------
    # [核心] 模型配置
    # ----------------------------------
    "models": {
        "Qwen_Base": {
            "enabled": True,
            "type": "local",
            "path": "./unsloth/Qwen3-4B-Instruct-2507",
            "color": "blue"
        },
        "Finetuned_Gen50": {
            "enabled": True,
            "type": "local",
            "path": "./best_model_gen_50",
            "color": "green"
        },
        "Finetuned_Gen01": {
            "enabled": True,
            "type": "local",
            "path": "./best_model_gen_01",
            "color": "yellow"
        },
        "DeepSeek_V3": {
            "enabled": True,
            "type": "api",
            "base_url": "https://api.deepseek.com",
            "api_key": "sk-822ab93a4fe24552bd17aedeb5b44562",
            "model_name": "deepseek-chat",
            "color": "red"
        }
    },

    # ----------------------------------
    # [核心] 评测参数
    # ----------------------------------
    "device": "cuda",
    "batch_size": 16,
    "max_new_tokens": 512,
    
    # 每个数据集最大样本数 (None=全量)
    "test_limit": 100, 
    
    "raw_data_dir": "./datasets",
    "extract_dir": "./datasets/extracted",
    "cache_dir": "./processed_cache",
    
    # 自动匹配关键词
    "benchmarks": {
        "ToMATO":      ["tomato"],
        "AwareLLM_SAD":["sad", "awareness"],
        "Mistake":     ["mistake", "bigbench"],
        "TreeCut":     ["treecut"],
        "HaloQuest":   ["haloquest"],
        "KG-FPQ":      ["kg-fpq", "fpq"],
        "CPED":        ["cped"],
        "SuperGPQA":   ["supergpqa", "gpqa"],
        "RBench":      ["rbench"],
        "SFE":         ["sfe"],
        "ExploreToM":  ["exploretom"],
        "LifeState":   ["lifestate"],
        "DIKWP":       ["dikwp"],
        "IRI_BES":     ["iri", "bes"],
        "GRAB":        ["grab"]
    }
}

# ==========================================
# 2. 智能数据解析器 (Data Processor)
# ==========================================
import json
import glob
import zipfile

class DataProcessor:
    def __init__(self):
        self.search_root = CONFIG['extract_dir']
        if not os.path.exists(CONFIG['cache_dir']): os.makedirs(CONFIG['cache_dir'])

    def find_best_file(self, keywords):
        candidates = []
        for root, dirs, files in os.walk(self.search_root):
            if any(x in root.lower() for x in ['__pycache__', '.git', 'package']): continue
            # 路径或文件夹名匹配
            if not (any(k in os.path.basename(root).lower() for k in keywords) or 
                    any(k in os.path.relpath(root, self.search_root).lower() for k in keywords)):
                continue
            
            for f in files:
                f_lower = f.lower()
                if f_lower.endswith(('.jsonl', '.json', '.csv', '.parquet')):
                    if 'ids' in f_lower or 'idx' in f_lower: continue
                    priority = 0
                    if 'test' in f_lower: priority += 3
                    elif 'eval' in f_lower: priority += 2
                    if f_lower.endswith('.jsonl'): priority += 0.5
                    candidates.append((priority, os.path.join(root, f)))
        
        candidates.sort(key=lambda x: x[0], reverse=True)
        return candidates[0][1] if candidates else None

    def normalize_entry(self, entry, bench_name=""):
        if not isinstance(entry, dict): return None
        keys = {k.lower(): k for k in entry.keys()}

        # CPED (情感分析)
        if 'cped' in bench_name.lower():
            u_key = next((k for k in ['utterance', 'sentence', 'text'] if k in keys), None)
            e_key = next((k for k in ['emotion', 'label', 'sentiment'] if k in keys), None)
            if u_key and e_key:
                return {"prompt": f"Analyze the emotion.\nSentence: \"{entry[keys[u_key]]}\"\nAnswer:", "gold": str(entry[keys[e_key]])}

        # ToMATO (心智理论)
        if 'tomato' in bench_name.lower() or ('conversation' in keys and 'a0' in keys):
            ctx = entry.get(keys.get('conversation', ''), '') or entry.get(keys.get('story', ''), '')
            q = entry.get(keys.get('q', ''), '')
            if not q: return None
            opts = []
            for i in range(5):
                k = f'a{i}'
                if k in keys: opts.append(f"{chr(65+i)}. {entry[keys[k]]}")
            opts_str = "\nOptions:\n" + "\n".join(opts)
            gold_idx = entry.get(keys.get('a_idx', ''), '')
            gold = chr(65 + int(gold_idx)) if str(gold_idx).isdigit() else str(entry.get(keys.get('a_str', ''), ''))
            return {"prompt": f"Context:\n{ctx}\n\nQuestion: {q}{opts_str}\nAnswer:", "gold": gold}

        # 通用匹配
        context = ""
        ctx_key = next((k for k in ['story', 'context', 'passage', 'text', 'dialogue'] 
                        if k in keys and isinstance(entry[keys[k]], str) and len(entry[keys[k]]) > 50), None)
        if ctx_key: context = f"Context:\n{entry[keys[ctx_key]]}\n\n"

        q_candidates = ['question', 'input', 'prompt', 'q', 'problem', 'utterance', 'tpq']
        if ctx_key in q_candidates: q_candidates.remove(ctx_key)
        q_key = next((k for k in q_candidates if k in keys), None)
        
        # HaloQuest 回退
        if not q_key and 'text' in keys and ctx_key != 'text' and len(str(entry[keys['text']])) < 300: q_key = 'text'
        if not q_key: return None

        opts_str = ""
        opt_key = next((k for k in ['options', 'choices'] if k in keys), None)
        if opt_key:
            opts = entry[keys[opt_key]]
            if isinstance(opts, list): opts_str = "\nOptions:\n" + "\n".join([f"{i}. {o}" for i, o in enumerate(opts)])
            elif isinstance(opts, dict): opts_str = "\nOptions:\n" + "\n".join([f"{k}. {v}" for k, v in opts.items()])

        a_key = next((k for k in ['answer', 'label', 'gold', 'output', 'target', 'emotion'] if k in keys), None)
        if not a_key: return None
        
        gold = str(entry[keys[a_key]])
        if gold.startswith("['") and gold.endswith("']"): gold = gold[2:-2]

        return {"prompt": f"{context}Question: {entry[keys[q_key]]}{opts_str}\nAnswer:", "gold": gold}

    def process(self, bench_name, keywords):
        cache_path = os.path.join(CONFIG['cache_dir'], f"{bench_name}.jsonl")
        if os.path.exists(cache_path): return cache_path

        raw_file = self.find_best_file(keywords)
        if not raw_file: return None
        
        data = []
        try:
            entries = []
            if raw_file.endswith('.csv'):
                try: df = pd.read_csv(raw_file); entries = df.to_dict('records')
                except: pass
            elif raw_file.endswith('.jsonl') or raw_file.endswith('.json'):
                with open(raw_file, 'r', encoding='utf-8') as f:
                    if raw_file.endswith('.jsonl'):
                        for line in f:
                            try: entries.append(json.loads(line))
                            except: pass
                    else:
                        content = json.load(f)
                        entries = content if isinstance(content, list) else []

            for entry in entries:
                processed = self.normalize_entry(entry, bench_name)
                if processed: data.append(processed)
            
            if len(data) > 0:
                print(f"   [Data] {bench_name}: Parsed {len(data)} samples.")
                with open(cache_path, 'w', encoding='utf-8') as f:
                    for d in data: f.write(json.dumps(d, ensure_ascii=False) + '\n')
                return cache_path
            return None
        except Exception: return None

# ==========================================
# 3. 评估器 (Evaluators)
# ==========================================
class BaseEvaluator:
    def extract_answer(self, pred):
        match = re.search(r'(?:^|answer:\s*|is\s+)([\(]?([a-e])[\)]?)(?:\.|,|\s|$)', pred.lower())
        if match: return match.group(2)
        numbers = re.findall(r'-?\d+\.?\d*', pred)
        return numbers[-1] if numbers else pred.lower()

    def check_correctness(self, pred, gold):
        gold_str = str(gold).lower().strip()
        pred_extracted = self.extract_answer(pred).lower()
        if gold_str == pred_extracted: return 1
        if gold_str in pred.lower(): return 1
        try:
            if abs(float(gold_str) - float(pred_extracted)) < 0.01: return 1
        except: pass
        pos = ['yes', 'true', 'correct']
        neg = ['no', 'false', 'incorrect']
        if gold_str in pos and any(x in pred.lower() for x in pos): return 1
        if gold_str in neg and any(x in pred.lower() for x in neg): return 1
        return 0

class LocalEvaluator(BaseEvaluator):
    def __init__(self, model_path):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        print(f"   🚀 Loading Local: {model_path}...")
        self.device = CONFIG['device']
        if HAS_UNSLOTH:
            self.model, self.tokenizer = FastLanguageModel.from_pretrained(
                model_name=model_path, max_seq_length=4096, dtype=None, load_in_4bit=True)
            FastLanguageModel.for_inference(self.model)
        else:
            self.tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side='left', trust_remote_code=True)
            self.model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True)
        
        self.tokenizer.padding_side = 'left'
        if not self.tokenizer.pad_token: self.tokenizer.pad_token = self.tokenizer.eos_token

    def generate(self, prompts):
        formatted = [f"<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n" for p in prompts]
        inputs = self.tokenizer(formatted, return_tensors="pt", padding=True, truncation=True).to(self.device)
        if inputs.input_ids.shape[1] > 3500: return ["Input too long"] * len(prompts)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, max_new_tokens=CONFIG['max_new_tokens'],
                pad_token_id=self.tokenizer.pad_token_id, eos_token_id=self.tokenizer.eos_token_id,
                do_sample=False, temperature=0.0
            )
        return [self.tokenizer.decode(out[inputs.input_ids.shape[1]:], skip_special_tokens=True).strip() for out in outputs]

    def unload(self):
        # 强制清理显存
        print("   🧹 Releasing GPU Memory...")
        if hasattr(self, 'model'):
            del self.model
        if hasattr(self, 'tokenizer'):
            del self.tokenizer
        
        # 循环清理以确保 Python 的引用计数归零
        for _ in range(3):
            gc.collect()
            torch.cuda.empty_cache()
        
        # 休息一下，让 GPU 驱动缓一缓
        time.sleep(5)
        print("   ✅ GPU Memory Cleared.")

class APIEvaluator(BaseEvaluator):
    def __init__(self, conf):
        print(f"   ☁️  API Connect: {conf['base_url']}")
        self.client = OpenAI(api_key=conf['api_key'], base_url=conf['base_url'])
        self.model_name = conf['model_name']

    def generate(self, prompts):
        results = []
        for p in prompts:
            try:
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[{"role": "user", "content": p}],
                    max_tokens=CONFIG['max_new_tokens'], temperature=0.0
                )
                results.append(response.choices[0].message.content.strip())
            except Exception as e:
                results.append(f"API Error: {str(e)}")
        return results
    
    def unload(self): pass

# ==========================================
# 4. 主流程
# ==========================================
def main():
    # 1. 准备数据
    if not os.path.exists(CONFIG['extract_dir']): os.makedirs(CONFIG['extract_dir'])
    zips = glob.glob(os.path.join(CONFIG['raw_data_dir'], "*.zip"))
    if zips and not os.listdir(CONFIG['extract_dir']):
        print("📦 Unzipping files...")
        for z in zips:
            try:
                with zipfile.ZipFile(z, 'r') as zf: zf.extractall(CONFIG['extract_dir'])
            except: pass

    processor = DataProcessor()
    tasks = []
    print("\n🔎 Checking Datasets...")
    for name, keywords in CONFIG['benchmarks'].items():
        path = processor.process(name, keywords)
        if path: tasks.append((name, path))

    if not tasks:
        print("❌ No datasets found. Check ./datasets")
        return

    # 2. 评测
    all_results = {} 
    
    print(f"\n🏁 Evaluation Start (Limit: {CONFIG['test_limit']})")
    
    for model_name, conf in CONFIG['models'].items():
        if not conf['enabled']: continue
        
        print(f"\n{'='*40}\n▶️  MODEL: {model_name}\n{'='*40}")
        
        try:
            evaluator = LocalEvaluator(conf['path']) if conf['type'] == 'local' else APIEvaluator(conf)
        except Exception as e:
            print(f"❌ Load Error: {e}")
            # 如果加载失败，也要尝试清理一下
            gc.collect()
            torch.cuda.empty_cache()
            continue

        model_scores = {}
        for bench_name, path in tasks:
            with open(path, 'r', encoding='utf-8') as f:
                dataset = [json.loads(line) for line in f]
            
            if CONFIG['test_limit'] and len(dataset) > CONFIG['test_limit']:
                dataset = random.sample(dataset, CONFIG['test_limit'])
            if conf['type'] == 'api' and len(dataset) > 10: 
                dataset = dataset[:10]
            
            # [Fix] 防止空数据集导致除以零
            if len(dataset) == 0:
                model_scores[bench_name] = 0.0
                continue

            correct = 0
            bs = 1 if conf['type'] == 'api' else CONFIG['batch_size']
            pbar = tqdm(range(0, len(dataset), bs), desc=f"{bench_name:<12}", colour=conf.get('color', 'white'))
            
            for i in pbar:
                batch = dataset[i : i + bs]
                preds = evaluator.generate([x['prompt'] for x in batch])
                for pred, gold in zip(preds, [x['gold'] for x in batch]):
                    correct += evaluator.check_correctness(pred, gold)
                
                # [Fix] 安全计算
                total_so_far = i + len(batch)
                acc = (correct / total_so_far * 100) if total_so_far > 0 else 0
                pbar.set_postfix({"Acc": f"{acc:.1f}%"})
            
            model_scores[bench_name] = (correct / len(dataset)) * 100 if len(dataset) > 0 else 0.0
        
        all_results[model_name] = model_scores
        
        # [Fix] 显式卸载模型 + 销毁对象
        evaluator.unload()
        del evaluator
        
        # 再次强制清理
        gc.collect()
        torch.cuda.empty_cache()

    # 3. 输出 Excel
    print("\n\n📊 Generating Excel Report...")
    try:
        df = pd.DataFrame(all_results)
        df.index.name = "Benchmark"
        
        if not df.empty:
            avg_row = df.mean()
            df.loc['AVERAGE'] = avg_row
            df = df.sort_index() 
        
        output_file = "benchmark_comparison_results.xlsx"
        df.to_excel(output_file)
        
        print("="*60)
        print(df.round(2))
        print("="*60)
        print(f"✅ Saved to: {os.path.abspath(output_file)}")
        
    except Exception as e:
        print(f"❌ Excel Error: {e}")
        print("Fallback to CSV...")
        df.to_csv("benchmark_comparison_results.csv")

if __name__ == "__main__":
    main()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Unsloth detected (Acceleration enabled).

🔎 Checking Datasets...

🏁 Evaluation Start (Limit: 100)

▶️  MODEL: Qwen_Base
   🚀 Loading Local: ./unsloth/Qwen3-4B-Instruct-2507...
==((====))==  Unsloth 2025.10.6: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.367 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+00a7a5f.d20251020. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

SuperGPQA   : 100%|██████████| 7/7 [03:09<00:00, 27.09s/it, Acc=22.0%]


   🧹 Releasing GPU Memory...
   ✅ GPU Memory Cleared.

▶️  MODEL: Finetuned_Gen50
   🚀 Loading Local: ./best_model_gen_50...
==((====))==  Unsloth 2025.10.6: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.367 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+00a7a5f.d20251020. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

SuperGPQA   : 100%|██████████| 7/7 [04:09<00:00, 35.66s/it, Acc=14.0%]


   🧹 Releasing GPU Memory...
   ✅ GPU Memory Cleared.

▶️  MODEL: DeepSeek_V3
   ☁️  API Connect: https://api.deepseek.com


SuperGPQA   : 100%|██████████| 10/10 [02:10<00:00, 13.01s/it, Acc=20.0%]




📊 Generating Excel Report...
              Qwen_Base  Finetuned_Gen50  Finetuned_Gen01  DeepSeek_V3
Benchmark                                                             
AVERAGE           34.67             32.0            32.33        38.89
AwareLLM_SAD      97.00             95.0            89.00       100.00
CPED              14.00             15.0            20.00        10.00
GRAB               0.00              0.0             0.00         0.00
HaloQuest          0.00              0.0             0.00         0.00
KG-FPQ             0.00              0.0             0.00         0.00
Mistake           54.00             39.0            45.00        50.00
SuperGPQA         22.00             14.0            14.00        20.00
ToMATO            85.00             86.0            83.00       100.00
TreeCut           40.00             39.0            40.00        70.00
✅ Saved to: /root/autodl-tmp/benchmark_comparison_results.xlsx
